# 🔥 Chapter 8 — Saving, Loading & Model Deployment

---

### 🎯 What you'll learn
- Saving and loading model weights (`state_dict`)
- Saving complete checkpoints (for resuming training)
- Exporting models (TorchScript, ONNX)
- ⚠️ Corner cases: loading on different devices, architecture mismatches

## 8.1 — Saving & Loading `state_dict` (Recommended)

In [ ]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 3)
    
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = MyModel()

# ✅ Save only the state_dict (weights + biases)
torch.save(model.state_dict(), 'model_weights.pth')
print("Model weights saved!")

# ✅ Load into a new model instance
model2 = MyModel()  # Must define architecture first!
model2.load_state_dict(torch.load('model_weights.pth'))
model2.eval()  # Set to eval mode for inference
print("Model weights loaded!")

## 8.2 — ⚠️ Corner Case: Loading on a Different Device

In [ ]:
# ⚠️ If model was saved on GPU but loading on CPU:
# model.load_state_dict(torch.load('model.pth'))  # ❌ May crash!

# ✅ Always specify map_location
model3 = MyModel()
model3.load_state_dict(
    torch.load('model_weights.pth', map_location='cpu')  # Force load to CPU
)
print("Loaded with map_location='cpu'")

# If loading to GPU:
# torch.load('model.pth', map_location='cuda:0')

# For device-agnostic loading:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# torch.load('model.pth', map_location=device)

## 8.3 — Full Checkpointing (Resume Training)

In [ ]:
# Save everything needed to resume training
model = MyModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': 0.05,
    'val_loss': 0.08,
}
torch.save(checkpoint, 'checkpoint.pth')
print("Checkpoint saved!")

# Resume training
model_resumed = MyModel()
optimizer_resumed = torch.optim.Adam(model_resumed.parameters())

checkpoint = torch.load('checkpoint.pth', map_location='cpu')
model_resumed.load_state_dict(checkpoint['model_state_dict'])
optimizer_resumed.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch']
print(f"Resumed from epoch {start_epoch}")

## 8.4 — ⚠️ Corner Case: `strict=False` for Partial Loading

In [ ]:
# When model architecture changed (e.g., added new layers)
class MyModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 3)
        self.fc3 = nn.Linear(3, 1)  # NEW layer!
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

model_v2 = MyModelV2()

# ⚠️ strict=True (default) will FAIL because fc3 is missing from saved weights
# model_v2.load_state_dict(torch.load('model_weights.pth'))  # ❌ Error!

# ✅ strict=False loads matching keys, ignores missing/unexpected
incompatible = model_v2.load_state_dict(
    torch.load('model_weights.pth', map_location='cpu'),
    strict=False
)
print(f"Missing keys:    {incompatible.missing_keys}")
print(f"Unexpected keys: {incompatible.unexpected_keys}")

## 8.5 — TorchScript: Export for Production

In [ ]:
model = MyModel()
model.eval()

# Method 1: Tracing (for models without control flow)
example_input = torch.randn(1, 10)
traced = torch.jit.trace(model, example_input)
traced.save('model_traced.pt')

# Load without needing the class definition!
loaded = torch.jit.load('model_traced.pt')
output = loaded(example_input)
print(f"TorchScript output: {output.shape}")

# ⚠️ Corner Case: tracing doesn't capture if/else branches!
# Use torch.jit.script() for models with dynamic control flow.

## ✏️ Exercises

In [ ]:
# Exercise 1: Write a function save_best_model(model, val_loss, best_loss, path)
# that only saves when val_loss improves.
# YOUR CODE HERE


# Exercise 2: Save a checkpoint, modify the model by removing a layer,
# load with strict=False, and verify which keys were missing.
# YOUR CODE HERE
